# Public LB 1.00 with Five Empty Masks

## Responsible disclosure: please fix the leaderboard

This is **not a segmentation solution**. It contains no trained model, no hidden labels, no external ground truth, and no predicted filament pixels. Its only purpose is to provide a minimal, reproducible demonstration that the current public evaluator can award a perfect score to an invalid solution.

The generated submission contains only **5 of the 180 test images** and assigns one completely empty mask to each. The total predicted foreground area is exactly zero. A foreground-aware instance Dice evaluation with missing-image penalties must give this submission a score of 0, but Kaggle returned **Public Score 1.00** for submission `55041507`.

I am publishing this immediately so participants do not spend compute optimizing against a broken leaderboard and so the organizers can repair the evaluator before the competition proceeds further. I am happy for this diagnostic submission to be invalidated after the issue is confirmed.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils

TEST_DIR = Path('/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images')
EXPECTED_IMAGE_SIZE = (2048, 2048)
OBSERVED_PUBLIC_SCORE = 1.00
SUBMISSION_REFERENCE = 55041507

assert TEST_DIR.exists(), f'Missing competition test data: {TEST_DIR}'
all_test_ids = {path.stem for path in TEST_DIR.glob('*.jpeg')}
assert len(all_test_ids) == 180, f'Expected 180 test images, found {len(all_test_ids)}'
print('Test images:', len(all_test_ids))

## Five-image diagnostic subset

These five IDs were selected without ground truth, using the lowest predicted foreground area from our own previously trained model. The selection is fixed below solely to reproduce the verified submission byte-for-byte. The vulnerability is that 175 omitted test images receive no penalty and empty masks can receive perfect credit.

In [ ]:
selected_test_ids = [
    '20151204133714Th',
    '20200310113230Ch',
    '20211214174010Mh',
    '20210607142350Bh',
    '20180507140210Th',
]
assert len(selected_test_ids) == 5
assert set(selected_test_ids).issubset(all_test_ids)

empty_512 = np.zeros((512, 512), dtype=np.uint8)
empty_rle = mask_utils.encode(np.asfortranarray(empty_512))
empty_counts = empty_rle['counts'].decode('ascii')
assert empty_counts == 'PPP8'
assert int(mask_utils.area(empty_rle)) == 0
assert int(mask_utils.decode(empty_rle).sum()) == 0

submission = pd.DataFrame({
    'filament_id': [f'{image_id}_1' for image_id in selected_test_ids],
    'segmentation_rle': [empty_counts] * len(selected_test_ids),
})

decoded_areas_at_expected_size = [
    int(mask_utils.area({'size': list(EXPECTED_IMAGE_SIZE), 'counts': value.encode('ascii')}))
    for value in submission['segmentation_rle']
]
assert sum(decoded_areas_at_expected_size) == 0
assert submission['filament_id'].is_unique
submission.to_csv('submission.csv', index=False)

print(submission.to_string(index=False))
print('Rows submitted:', len(submission), '/ 180 test images')
print('Total predicted foreground pixels:', sum(decoded_areas_at_expected_size))
print('Expected foreground Dice:', 0.0)
print('Observed public score:', OBSERVED_PUBLIC_SCORE)

## Verified result

| Property | Value |
|---|---:|
| Trained model | None |
| Test images provided | 5 / 180 |
| Masks provided | 5 |
| Predicted foreground pixels | 0 |
| Correct foreground Dice | 0.00 |
| Kaggle public score | **1.00** |
| Submission reference | `55041507` |

Earlier controls strengthen the diagnosis: one empty mask for every test image scored 0.88, while seven empty masks per image scored 0.70. Restricting the file to a small subset produced a perfect 1.00. This result cannot represent segmentation quality.

## Suggested evaluator safeguards

1. Require coverage of all 180 test image IDs and reject incomplete submissions.
2. Reject zero-area predicted instances.
3. Validate that every RLE count stream represents exactly 2048×2048 pixels.
4. Compute Dice on foreground instances only; do not reward background true negatives.
5. Assign zero to every unmatched ground-truth instance and every missing test image.
6. Penalize unmatched extra predictions and fragmentation explicitly.
7. Add tests for empty, partial, all-full, perfect, missing-instance, and excessive-fragmentation submissions.

In [ ]:
summary = {
    'purpose': 'responsible disclosure',
    'submission_reference': SUBMISSION_REFERENCE,
    'observed_public_score': OBSERVED_PUBLIC_SCORE,
    'test_images_total': len(all_test_ids),
    'test_images_submitted': len(selected_test_ids),
    'test_images_missing': len(all_test_ids) - len(selected_test_ids),
    'submission_rows': len(submission),
    'unique_rles': int(submission['segmentation_rle'].nunique()),
    'rle': empty_counts,
    'total_predicted_foreground_pixels': int(sum(decoded_areas_at_expected_size)),
    'correct_foreground_dice': 0.0,
    'conclusion': 'The current public leaderboard can award a perfect score to an incomplete all-empty submission.',
}
Path('responsible_disclosure_summary.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)
print(json.dumps(summary, indent=2))